# KAN-REC — Entrenamiento completo con 8M filas (Google Colab)

**TFM · Máster en Big Data & Data Engineering · UCM**  
Autor: Pedro Antonio Martínez Sánchez

## Requisitos previos
1. Subir `criteo_10m.tsv` a Google Drive en `/kanrec/criteo_10m.tsv`
2. Activar GPU: Entorno de ejecución → Cambiar tipo → GPU (T4 o A100)
3. Ejecutar celdas en orden

## Estructura de salida
```
Drive/kanrec/
├── criteo_10m.tsv              ← subir antes de empezar
├── feature_selection.json      ← generado en celda 3
├── checkpoints/
│   ├── best_kan-bspline_8M_s42.pt
│   ├── best_kan-bspline_8M_s123.pt
│   └── best_kan-bspline_8M_s256.pt
└── results/
    ├── training_results_8M.json
    └── symbolic_results_8M.json
```

In [ ]:
# ── Celda 1: Instalar dependencias ────────────────────────────────────────────
!pip install -q efficient-kan
!pip install -q numpy==1.26.4

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Celda 2: Montar Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/kanrec'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

CSV_PATH = f'{DRIVE_BASE}/criteo_10m.tsv'
assert os.path.exists(CSV_PATH), f'ERROR: No se encuentra {CSV_PATH}. Sube el CSV a Drive primero.'
size_gb = os.path.getsize(CSV_PATH) / 1024**3
print(f'criteo_10m.tsv encontrado: {size_gb:.2f} GB')

In [ ]:
# ── Celda 3: Preprocesado con pandas (replica Spark MLlib Pipeline) ───────────
#
# IMPORTANTE: Este preprocesado replica exactamente el Pipeline MLlib de Fabric:
#   - log1p en columnas I1-I5 (contadores, cola larga)
#   - StandardScaler en columnas I6-I13 (gaussianas)
#   - Split 80/10/10 con seed=42
#   - Ajuste de estadísticos SOLO sobre train (sin data leakage)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import json, time

NUMERICAL_COLS   = [f'I{i}' for i in range(1, 14)]
CATEGORICAL_COLS = [f'C{i}' for i in range(1, 27)]
LOG_COLS = [f'I{i}' for i in range(1, 6)]
STD_COLS = [f'I{i}' for i in range(6, 14)]
ALL_COLS = ['label'] + NUMERICAL_COLS + CATEGORICAL_COLS

print('Cargando criteo_10m.tsv en chunks...')
t0 = time.time()

# Leer en chunks para no saturar RAM
CHUNK_SIZE = 500_000
chunks = []
reader = pd.read_csv(
    CSV_PATH, sep='\t', header=None, names=ALL_COLS,
    chunksize=CHUNK_SIZE
)
for i, chunk in enumerate(reader):
    # Limpiar nulos en numéricas
    for c in NUMERICAL_COLS:
        chunk[c] = chunk[c].fillna(0.0).clip(lower=0).astype('float32')
    # Limpiar nulos en categóricas
    for c in CATEGORICAL_COLS:
        chunk[c] = chunk[c].fillna('<UNK>')
    chunks.append(chunk)
    if (i+1) % 5 == 0:
        print(f'  Chunk {i+1}: {(i+1)*CHUNK_SIZE:,} filas leídas')

df = pd.concat(chunks, ignore_index=True)
print(f'Total: {len(df):,} filas en {time.time()-t0:.1f}s')
print(f'CTR global: {df["label"].mean():.4f}')

In [ ]:
# ── Celda 4: Split y normalización ───────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# Split 80/10/10 (mismo que Spark randomSplit seed=42)
np.random.seed(42)
idx = np.random.permutation(len(df))
n_train = int(len(df) * 0.80)
n_val   = int(len(df) * 0.90)

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_val]
test_idx  = idx[n_val:]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f'TRAIN: {len(train_df):,} | VAL: {len(val_df):,} | TEST: {len(test_df):,}')
del df  # liberar memoria

# log1p en columnas de cola larga (ajustado sobre train)
for c in LOG_COLS:
    train_df[c] = np.log1p(train_df[c])
    val_df[c]   = np.log1p(val_df[c])
    test_df[c]  = np.log1p(test_df[c])

# StandardScaler ajustado SOLO sobre train
scaler = StandardScaler()
train_df[STD_COLS] = scaler.fit_transform(train_df[STD_COLS]).astype('float32')
val_df[STD_COLS]   = scaler.transform(val_df[STD_COLS]).astype('float32')
test_df[STD_COLS]  = scaler.transform(test_df[STD_COLS]).astype('float32')
print('Normalización completada (sin data leakage)')

# Encoding categórico: LabelEncoder ajustado sobre train
encoders = {}
for c in CATEGORICAL_COLS:
    le = LabelEncoder()
    train_df[c] = le.fit_transform(train_df[c].astype(str))
    # Para val/test: valores desconocidos → 0
    le_classes = set(le.classes_)
    val_df[c]  = val_df[c].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le_classes else 0)
    test_df[c] = test_df[c].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le_classes else 0)
    encoders[c] = le

cat_cardinalities = [int(train_df[c].max()) + 10 for c in CATEGORICAL_COLS]
print(f'Encoding categórico completado: {len(CATEGORICAL_COLS)} campos')
print(f'Cardinalidades: min={min(cat_cardinalities)}, max={max(cat_cardinalities)}')

# Guardar feature_selection.json
selection = {'selected': NUMERICAL_COLS, 'excluded': [], 'cat_cardinalities': cat_cardinalities}
with open(f'{DRIVE_BASE}/feature_selection.json', 'w') as f:
    json.dump(selection, f, indent=2)
print('feature_selection.json guardado en Drive')

In [ ]:
# ── Celda 5: Dataset PyTorch ──────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class CriteoDataset(Dataset):
    def __init__(self, df, num_cols, cat_cols):
        self.x_num = torch.tensor(
            df[num_cols].values.astype('float32'), dtype=torch.float32)
        self.x_cat = torch.tensor(
            df[cat_cols].values.astype('int64'), dtype=torch.long)
        self.y = torch.tensor(
            df['label'].values.astype('float32'), dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x_num[i], self.x_cat[i], self.y[i]

print('Creando datasets completos...')
train_ds = CriteoDataset(train_df, NUMERICAL_COLS, CATEGORICAL_COLS)
val_ds   = CriteoDataset(val_df,   NUMERICAL_COLS, CATEGORICAL_COLS)
test_ds  = CriteoDataset(test_df,  NUMERICAL_COLS, CATEGORICAL_COLS)

BATCH_SIZE = 4096  # GPU permite batches más grandes
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

In [ ]:
# ── Celda 6: Modelo KAN-REC ───────────────────────────────────────────────────
import torch.nn as nn
from efficient_kan import KAN

class KANNumericalEncoder(nn.Module):
    def __init__(self, num_fields, embedding_dim=16, grid_size=5, spline_order=3):
        super().__init__()
        self.field_kans = nn.ModuleList([
            KAN(layers_hidden=[1, embedding_dim],
                grid_size=grid_size, spline_order=spline_order)
            for _ in range(num_fields)
        ])
    def forward(self, x):
        return torch.cat([kan(x[:, j:j+1]).unsqueeze(1)
                         for j, kan in enumerate(self.field_kans)], dim=1)
    def get_spline_curves(self, field_idx, n_points=300):
        x_grid = torch.linspace(-3.0, 3.0, n_points).unsqueeze(1)
        with torch.no_grad(): y = self.field_kans[field_idx](x_grid)
        return x_grid.squeeze(), y
    def get_edge_norms(self):
        return [sum(p.abs().sum().item() for p in kan.parameters())
                for kan in self.field_kans]


class KANRecModel(nn.Module):
    def __init__(self, num_numerical, cat_cardinalities,
                 embedding_dim=16, grid_size=5, spline_order=3):
        super().__init__()
        self.numerical_encoder = KANNumericalEncoder(
            num_numerical, embedding_dim, grid_size, spline_order)
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim, padding_idx=0)
            for card in cat_cardinalities])
        total_fields = num_numerical + len(cat_cardinalities)
        self.interaction = nn.Sequential(
            nn.Linear(total_fields * embedding_dim, 256),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 64), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x_num, x_cat):
        num_emb = self.numerical_encoder(x_num)
        cat_embs = [emb(x_cat[:, i].clamp(0, emb.num_embeddings-1))
                    for i, emb in enumerate(self.cat_embeddings)]
        all_emb = torch.cat([num_emb, torch.stack(cat_embs, dim=1)], dim=1)
        return self.head(self.interaction(all_emb.view(all_emb.size(0), -1)))

    def entropy_reg_loss(self):
        reg = torch.tensor(0.0, device=next(self.parameters()).device)
        for kan in self.numerical_encoder.field_kans:
            if hasattr(kan, 'spline_weight'):
                w = kan.spline_weight.abs()
                w_norm = w / (w.sum() + 1e-8)
                reg += -(w_norm * (w_norm + 1e-8).log()).sum()
        return 1e-3 * reg


model = KANRecModel(
    num_numerical=len(NUMERICAL_COLS),
    cat_cardinalities=cat_cardinalities,
    embedding_dim=16, grid_size=5, spline_order=3,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'KANRecModel creado: {total_params:,} parámetros | Device: {device}')

In [ ]:
# ── Celda 7: Entrenamiento completo 8M filas ──────────────────────────────────
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, log_loss
import time, json

SEEDS = [42, 123, 256]
all_results = []

for SEED in SEEDS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = KANRecModel(
        len(NUMERICAL_COLS), cat_cardinalities, 16, 5, 3).to(device)

    optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion  = nn.BCELoss()
    best_val_auc, patience_ctr = 0.0, 0
    MAX_EPOCHS, PATIENCE = 10, 3
    CKPT_FILE = f'{DRIVE_BASE}/checkpoints/best_kan-bspline_8M_s{SEED}.pt'

    print(f"\n{'='*60}")
    print(f'Entrenando KAN-REC 8M filas | seed={SEED}')
    print(f'Dataset: {len(train_ds):,} filas | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
    print(f"{'='*60}")

    for epoch in range(MAX_EPOCHS):
        t_epoch = time.time()

        # Train
        model.train()
        train_loss, n_batches = 0.0, 0
        for x_num, x_cat, y in train_loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            y     = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            pred = model(x_num, x_cat).squeeze()
            loss = criterion(pred, y) + model.entropy_reg_loss()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            n_batches  += 1

        # Validate
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                p = model(x_num.to(device), x_cat.to(device)).squeeze().cpu().numpy()
                val_preds.extend(p); val_labels.extend(y.numpy())

        val_auc = roc_auc_score(val_labels, val_preds)
        val_ll  = log_loss(val_labels, val_preds)
        scheduler.step(1 - val_auc)
        elapsed = time.time() - t_epoch

        print(f'  Epoch {epoch+1:02d} | loss={train_loss/n_batches:.4f} | '
              f'val_auc={val_auc:.4f} | val_ll={val_ll:.4f} | {elapsed:.0f}s')

        if val_auc > best_val_auc:
            best_val_auc = val_auc; patience_ctr = 0
            torch.save(model.state_dict(), CKPT_FILE)
            print(f'    → Mejor modelo guardado en Drive (AUC={val_auc:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Early stopping epoch {epoch+1}'); break

    # Test
    model.load_state_dict(torch.load(CKPT_FILE, map_location=device))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_num, x_cat, y in test_loader:
            p = model(x_num.to(device), x_cat.to(device)).squeeze().cpu().numpy()
            test_preds.extend(p); test_labels.extend(y.numpy())

    test_auc = roc_auc_score(test_labels, test_preds)
    test_ll  = log_loss(test_labels, test_preds)

    result = {
        'encoder': 'kan-bspline', 'dataset': 'criteo-8M',
        'seed': SEED, 'test_auc': test_auc, 'test_logloss': test_ll,
        'checkpoint': CKPT_FILE,
    }
    all_results.append(result)

    print(f"\n  TEST AUC={test_auc:.4f} | Log-loss={test_ll:.4f}")
    print(f'  Checkpoint: {CKPT_FILE}')

# Resumen final
print(f"\n{'='*60}")
print('RESULTADOS KAN-REC 8M filas:')
for r in all_results:
    print(f"  Seed {r['seed']}: AUC={r['test_auc']:.4f} | LL={r['test_logloss']:.4f}")
aucs = [r['test_auc'] for r in all_results]
print(f"  Media: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# Guardar resultados en Drive
with open(f'{DRIVE_BASE}/results/training_results_8M.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\nResultados guardados en Drive.')

In [ ]:
# ── Celda 8: Extracción simbólica post-entrenamiento ──────────────────────────
from scipy.optimize import curve_fit
from collections import Counter

OPERATOR_LIBRARY = {
    'log':     lambda x, a, b: a * np.log(np.abs(x) + 1) + b,
    'exp':     lambda x, a, b: a * np.exp(np.clip(x, -10, 10)) + b,
    'square':  lambda x, a, b: a * x**2 + b,
    'sqrt':    lambda x, a, b: a * np.sqrt(np.abs(x)) + b,
    'sigmoid': lambda x, a, b: a / (1 + np.exp(-x)) + b,
    'linear':  lambda x, a, b: a * x + b,
}
FORMULA_TEMPLATES = {
    'log':    '{a}·log(|x|+1) + {b}',
    'exp':    '{a}·exp(x) + {b}',
    'square': '{a}·x² + {b}',
    'sqrt':   '{a}·√|x| + {b}',
    'sigmoid':'{a}·σ(x) + {b}',
    'linear': '{a}·x + {b}',
}

def fit_field(model, field_idx, r2_threshold=0.90):
    x_grid, y_curves = model.numerical_encoder.get_spline_curves(field_idx)
    x_np, y_np = x_grid.numpy(), y_curves[:, 0].numpy()
    best = {'operator': None, 'r2': -1.0, 'params': None, 'formula': '?', 'accepted': False}
    for name, fn in OPERATOR_LIBRARY.items():
        try:
            params, _ = curve_fit(fn, x_np, y_np, maxfev=5000)
            y_pred = fn(x_np, *params)
            r2 = float(1 - np.sum((y_np-y_pred)**2) / (np.sum((y_np-y_np.mean())**2)+1e-10))
            if r2 > best['r2']:
                a, b = round(float(params[0]),4), round(float(params[1]),4)
                best = {'operator': name, 'r2': r2, 'params': [float(params[0]), float(params[1])],
                        'formula': FORMULA_TEMPLATES[name].format(a=a,b=b), 'accepted': r2 >= r2_threshold}
        except: continue
    return best

print('Extracción simbólica para las 3 semillas...')
results_symbolic = {}

for SEED in SEEDS:
    ckpt = f'{DRIVE_BASE}/checkpoints/best_kan-bspline_8M_s{SEED}.pt'
    if not os.path.exists(ckpt): print(f'Falta: {ckpt}'); continue

    state_dict = torch.load(ckpt, map_location='cpu')
    cards = [state_dict[f'cat_embeddings.{i}.weight'].shape[0] for i in range(26)]
    m = KANRecModel(len(NUMERICAL_COLS), cards, 16, 5, 3)
    m.load_state_dict(state_dict); m.eval()

    norms = m.numerical_encoder.get_edge_norms()
    threshold = np.percentile(norms, 20)
    surviving = [j for j, n in enumerate(norms) if n >= threshold]

    print(f'\nSeed {SEED}: {len(surviving)}/{len(norms)} campos supervivientes')
    seed_results = {}
    for j in surviving:
        r = fit_field(m, j)
        field = NUMERICAL_COLS[j]
        seed_results[field] = r
        status = '✓' if r['accepted'] else '✗'
        print(f'  {status} {field}: {r["operator"]:8s} R²={r["r2"]:.4f}  φ(x) ≈ {r["formula"]}')
    results_symbolic[SEED] = seed_results

# Análisis de estabilidad
print(f"\n{'='*55}")
print('ANÁLISIS DE ESTABILIDAD SIMBÓLICA:')
field_ops = {}
for seed, results in results_symbolic.items():
    for field, r in results.items():
        if r['accepted']: field_ops.setdefault(field, []).append(r['operator'])

for field, ops in sorted(field_ops.items()):
    dominant, count = Counter(ops).most_common(1)[0]
    print(f'  {field:<6}: {dominant:<10} {count}/3 seeds ({count/3:.0%})')

# Guardar en Drive
with open(f'{DRIVE_BASE}/results/symbolic_results_8M.json', 'w') as f:
    json.dump({'results_by_seed': {str(k): v for k, v in results_symbolic.items()},
               'stability': {f: {'dominant': Counter(ops).most_common(1)[0][0],
                                  'stability': Counter(ops).most_common(1)[0][1]/3}
                             for f, ops in field_ops.items()}}, f, indent=2)
print('\nsymbolic_results_8M.json guardado en Drive.')

In [ ]:
# ── Celda 9: Resumen final y comparativa ─────────────────────────────────────
print('='*60)
print('RESUMEN FINAL — KAN-REC entrenado con 8M filas reales')
print('='*60)

with open(f'{DRIVE_BASE}/results/training_results_8M.json') as f:
    results_8M = json.load(f)

print(f"\n{'Seed':<8} {'Test AUC':>10} {'Log-loss':>10}")
print('-'*32)
for r in results_8M:
    print(f"  {r['seed']:<6} {r['test_auc']:>10.4f} {r['test_logloss']:>10.4f}")

aucs = [r['test_auc'] for r in results_8M]
lls  = [r['test_logloss'] for r in results_8M]
print(f"  {'Media':<6} {np.mean(aucs):>10.4f} {np.mean(lls):>10.4f}")
print(f"  {'±Std':<6} {np.std(aucs):>10.4f}")

print(f"""
Comparativa actualizada:
  Raw normalisation (100K):  AUC ~0.885
  AutoDis    (100K, Fabric): AUC  0.8168
  KAN-REC    (100K, Fabric): AUC  0.8827
  KAN-REC    (8M,   Colab):  AUC  {np.mean(aucs):.4f} ← ESTE EXPERIMENTO

Próximo paso: descargar checkpoints de Drive y subirlos a Fabric OneLake.
""")